# 12 · Detect-and-gate on real CMU-MOSEI (short, CPU, no retraining)

The synthetic noisy-TV screen (`bench/`, 40+ techniques) found that the experts' own sigma cannot tell a corrupted
input from a clean one, and that **input-level detectors** can: an ELECTRA-style "which modality was replaced?"
classifier, input-density scores and a cross-modal predictability score. Here the best of them gate the
**frozen** PoE models from notebooks 03 / 07: a flagged modality is treated as missing for that clip.

**Held-out corruptions** (never generated by any detector): noise at λ = 0.5, half-clip noise, a gain fault,
a dead sensor, a time shuffle, two noisy modalities, and a swap with the most similar-looking clip.

Attach the outputs of **01** (experts + `mosei_compact.npz`), **03** (PoE models) and optionally **07**
(variational-target models). CPU is enough (about 30–40 min); GPU is faster.

In [ ]:
# Setup: make the `fusionlab` library importable. On Kaggle, always use a FRESH clone of the latest code (kept in /tmp,
# so it is not saved into this notebook's output); attached copies are used only when there is no Internet.
import glob, os, subprocess, sys

REPO_URL = 'https://github.com/fasihh/bigthing'      # the JF/ folder holds this project
BRANCH = 'jf-experiments'                           # used only until JF/ is merged into main


def fresh_clone(dest='/tmp/bigthing'):
    if not os.path.isdir(os.path.join(dest, '.git')):
        r = subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, dest], capture_output=True, text=True)
        if r.returncode:
            return None
    if not os.path.isdir(os.path.join(dest, 'JF')):  # JF not merged into main yet: use its branch
        subprocess.run(['git', '-C', dest, 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', dest, 'checkout', '-q', 'FETCH_HEAD'], check=True)
    return os.path.join(dest, 'JF')


def find_jf():
    here = os.path.abspath('.')
    if not here.startswith('/kaggle'):                # running inside a local checkout of the repo
        for c in (os.path.dirname(here), here, os.path.join(here, 'JF')):
            if os.path.isdir(os.path.join(c, 'src', 'fusionlab')):
                return c
    jf = fresh_clone()
    if jf:
        return jf
    for c in glob.glob('/kaggle/input/**/JF', recursive=True):   # offline fallback (may be older code)
        if os.path.isdir(os.path.join(c, 'src', 'fusionlab')):
            print('WARNING: no Internet, using an attached copy of the code:', c)
            return c
    raise RuntimeError('Cannot find the fusionlab code: turn on Internet in the notebook settings.')


JF = find_jf()
sys.path.insert(0, os.path.join(JF, 'src'))
CONFIGS = os.path.join(JF, 'configs')
commit = subprocess.run(['git', '-C', JF, 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('using fusionlab from', JF, '| commit', commit or 'n/a')

In [ ]:
import time
import numpy as np, pandas as pd
import torch
from fusionlab import BASE, config, data, gate, pretrain, report, runner
from fusionlab.runtime import RT

cfg = config.load_config(os.path.join(CONFIGS, '12_mosei_gate.yaml'))
data.load_mosei(cfg)
OUT = cfg['out_dir']

In [ ]:
def models():
    for obj in cfg['objectives']:
        for seed in cfg['seeds']:
            if not cfg['quick'] and not pretrain.find_ckpt(f'v2_model_main_{obj}_frozen_poe_s{seed}.pt'):
                print('skip (no checkpoint attached):', obj, seed); continue
            experts, _, marg = pretrain.pretrain_or_load(obj, BASE, 'main', seed, cfg)
            model, _ = runner.build_model(experts, obj, 'frozen', 'poe', BASE, cfg, seed, marg, 'main')
            yield obj, seed, model


tr_mask = {m: RT.DATA['train'][m + 'm'] for m in BASE}
feat_mean = {m: RT.DATA['train'][m][tr_mask[m]].mean(0).to(RT.DEV) for m in BASE}
swap_perm = torch.randperm(RT.N_SPLIT['test'], generator=torch.Generator().manual_seed(123 + 7))
conds = gate.gate_conditions(include_heldout=True)
print(len(conds), 'test conditions')

In [ ]:
rows = []
for obj, seed, model in models():
    t0 = time.time()
    clean = dict(name='clean', kind='clean')
    tr = gate.run_condition(model, 'train', clean, {}, feat_mean)
    trn = gate.run_condition(model, 'train', dict(name='trainnoise', kind='noise2', ms=BASE), {}, feat_mean)
    va = gate.run_condition(model, 'valid', clean, {}, feat_mean)
    te0 = gate.run_condition(model, 'test', clean, {}, feat_mean)
    perms = dict(swap=swap_perm, swapnn={m: gate.nn_perm(te0['F'], m) for m in BASE})
    dets = {'input_maha': gate.InputMaha().fit(tr), 'latent_maha': gate.LatentMaha().fit(tr),
            'crossmodal': gate.CrossModal().fit(tr),
            'electra': gate.Electra().fit(tr, trn['F'], seed=seed, epochs=cfg['electra_epochs'])}
    for d in dets.values():
        gate.calibrate(d, va)
    dets['combo:input_maha+crossmodal'] = gate.calibrate(gate.Combo([dets['input_maha'], dets['crossmodal']]), va)
    dets['combo:electra+crossmodal'] = gate.calibrate(gate.Combo([dets['electra'], dets['crossmodal']]), va)
    full = tuple(1.0 for _ in BASE)
    # variance temperature per technique on clean validation clips (same rule for every technique)
    tau = {'poe': np.mean((va['y'] - va['pred'][full][0]) ** 2 / va['pred'][full][1])}
    for k, d in dets.items():
        mu, v, _ = gate.gate_drop(va, d)
        tau[k] = np.mean((va['y'] - mu) ** 2 / v)
    clean_scores = {}
    for cd in conds:
        r = te0 if cd['kind'] == 'clean' else gate.run_condition(model, 'test', cd, perms, feat_mean)
        y = r['y']
        base = dict(objective=obj, seed=seed, cond=cd['name'], kind=cd['kind'], heldout=cd.get('heldout', False),
                    corrupted=cd.get('m', ''.join(cd.get('ms', ()))))
        outs = {'poe': (*r['pred'][full], np.exp(0.5 * np.stack([r['lv'][m].mean(1) for m in BASE], 1)))}
        if 'm' in cd:
            k = BASE.index(cd['m'])
            p = tuple(0.0 if j == k else 1.0 for j in range(len(BASE)))
            outs['oracle_drop'] = (*r['pred'][p], None)
        for k, d in dets.items():
            outs[k] = gate.gate_drop(r, d)
        for name, (mu, v, R) in outs.items():
            row = dict(base, technique=name, **gate.metrics(y, mu, v, tau.get(name, tau['poe'])))
            score = np.sqrt(v) if name in ('poe', 'oracle_drop') else R.max(1)
            if cd['kind'] == 'clean':
                clean_scores[name] = score
            elif name in clean_scores:
                row['AUROC'] = gate.auroc(clean_scores[name], score)
                if R is not None and 'm' in cd and name != 'poe':
                    row['loc_acc'] = np.mean(R.argmax(1) == BASE.index(cd['m']))
            rows.append(row)
    pd.DataFrame(rows).to_csv(os.path.join(OUT, 'gate_results.csv'), index=False)
    print(f'{obj} s{seed}: {time.time() - t0:.0f}s', flush=True)

R = pd.DataFrame(rows)
R.to_csv(os.path.join(OUT, 'gate_results.csv'), index=False)

## Results
`seen` = corruptions of the kind the ELECTRA detector generated for itself (white noise λ = 1, random swap);
`held-out` = everything else. Mean over seeds; one row per objective × technique.

In [ ]:
def board(R):
    corr = R[R.kind != 'clean']
    g = lambda df: df.groupby(['objective', 'technique', 'seed']).mean(numeric_only=True).groupby(['objective', 'technique'])
    out = pd.DataFrame({
        'clean_NLL': g(R[R.kind == 'clean'])['NLL'].mean(),
        'clean_MAE': g(R[R.kind == 'clean'])['MAE'].mean(),
        'seen_MAE': g(corr[~corr.heldout])['MAE'].mean(),
        'seen_NLL': g(corr[~corr.heldout])['NLL'].mean(),
        'heldout_MAE': g(corr[corr.heldout])['MAE'].mean(),
        'heldout_NLL': g(corr[corr.heldout])['NLL'].mean(),
        'heldout_Cov90': g(corr[corr.heldout])['Cov90'].mean(),
        'AUROC_seen': g(corr[~corr.heldout])['AUROC'].mean(),
        'AUROC_heldout': g(corr[corr.heldout])['AUROC'].mean(),
        'loc_acc': g(corr)['loc_acc'].mean(),
    })
    return out.round(3)

B = board(R)
B.to_csv(os.path.join(OUT, 'headline_12.csv'))
B

In [ ]:
# per held-out corruption kind: MAE (lower is better), PoE vs the gates
K = R[R.heldout].groupby(['objective', 'kind', 'technique']).MAE.mean().unstack('technique').round(3)
K.to_csv(os.path.join(OUT, 'scoreboard_12_kinds.csv'))
K

In [ ]:
# paired wins vs ungated PoE on corrupted NLL (per objective x seed)
corr = R[R.kind != 'clean'].groupby(['objective', 'seed', 'technique']).NLL.mean().unstack('technique')
wins = {t: f"{int((corr[t] < corr['poe']).sum())}/{len(corr)}" for t in corr.columns if t != 'poe'}
pd.Series(wins, name='beats PoE on corrupted NLL')